In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, when, lit
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType

# 1. Khởi tạo Spark Session & iceberg
spark = SparkSession.builder \
    .appName("silver_transfrom") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hadoop") \
    .config("spark.sql.catalog.iceberg.warehouse", "file:///data/lakehouse/iceberg") \
    .getOrCreate()


# Đọc trực tiếp từ bảng Iceberg tầng Bronze vừa tạo
df_bronze_loaded = spark.read.table("iceberg.bronze.taxi_trips_raw")

# Hiển thị 5 dòng đầu để kiểm tra chuỗi JSON thô
df_bronze_loaded.show(5, truncate=False)

print(spark.read.table("iceberg.bronze.taxi_trips_raw").columns)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/07 07:58:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+
|offset |partition|json_str                                                                                                                                                                                                                                                                                                                                                                                                                                                              |ingested_at            |
+-

In [6]:
taxi_schema = StructType([
     StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True)  
])

#read from bronze table
df_bronze = spark.read.table("iceberg.bronze.taxi_trips_raw").limit(1000)

#Parse json with the defined schema

df_parsed = df_bronze.withColumn("parsed_data", from_json(col("json_str"), taxi_schema)) \
    .select("parsed_data.*","ingested_at","offset","partition")

# 3. Gắn nhãn lỗi (Data Validation & Error Tagging)
df_validated = df_parsed.withColumn(
    "error_reason",
    when(col("tpep_pickup_datetime").isNull() | col("tpep_dropoff_datetime").isNull(), lit("NULL_TIMESTAMP"))
    .when(col("tpep_dropoff_datetime") <= col("tpep_pickup_datetime"), lit("INVALID_TIME_SEQUENCE"))
    .when(col("trip_distance") < 0, lit("NEGATIVE_DISTANCE"))
    .when(col("fare_amount") < 0, lit("NEGATIVE_FARE"))
    .when(col("total_amount") < 0, lit("NEGATIVE_TOTAL_AMOUNT"))
    .when(col("passenger_count").isNull() | (col("passenger_count") < 0), lit("INVALID_PASSENGER_COUNT"))
    .when(col("PULocationID").isNull() | col("DOLocationID").isNull(), lit("NULL_LOCATION"))
    .when(~col("payment_type").isin([0, 1, 2, 3, 4, 5, 6]), lit("INVALID_PAYMENT_TYPE"))
    .otherwise(lit(None)) # Không có lỗi -> Hợp lệ
)

# 4. Tách nhánh Clean và Quarantine
df_clean = df_validated.filter(col("error_reason").isNull()).drop("error_reason")
df_quarantine = df_validated.filter(col("error_reason").isNotNull())

# 5. Ghi xuống 2 bảng Iceberg tương ứng
df_clean.write \
    .format("iceberg") \
    .mode("append") \
    .saveAsTable("iceberg.silver.taxi_trips_cleaned")

df_quarantine.write \
    .format("iceberg") \
    .mode("append") \
    .saveAsTable("iceberg.silver.taxi_trips_quarantine")

print(f"Đã xử lý xong! Số dòng sạch: {df_clean.count()}, Số dòng lỗi vào Quarantine: {df_quarantine.count()}")




Đã xử lý xong! Số dòng sạch: 992, Số dòng lỗi vào Quarantine: 21


In [2]:

#print err data
df_quarantie_load = spark.read.table("iceberg.silver.taxi_trips_quarantine")

df_quarantie_load.show(truncate=False)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------------------+-------+---------+-------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|ingested_at            |offset |partition|error_reason |
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------------------+-------+---------+-------------+
|2       |2024-01-0

In [3]:
#print cleaned data

df_cleaned_load = spark.read.table("iceberg.silver.taxi_trips_cleaned")

df_cleaned_load.show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------------------+-------+---------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|ingested_at           |offset |partition|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------------------+-------+---------+
|2       |2024-01-03 12:45:59 |2024-01-03 12:52:16  |1.0        

In [ ]:
nhu